In [ ]:
import pandas as pd
import numpy as np
import os
import json
import gc  # Thư viện thu hồi bộ nhớ RAM chủ động
from datetime import datetime

# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN FILE DỮ LIỆU THỰC TẾ
# ==========================================
DATA_DIR = r"/content/DSS/archive"

paths = {
    'users': os.path.join(DATA_DIR, 'users_data.csv'),
    'cards': os.path.join(DATA_DIR, 'cards_data.csv'),
    'transactions': os.path.join(DATA_DIR, 'transactions_data.csv'),
    'labels': os.path.join(DATA_DIR, 'train_fraud_labels.json'),
    'mcc': os.path.join(DATA_DIR, 'mcc_codes.json')
}

# ==========================================
# HÀM TỐI ƯU HOÁ BỘ NHỚ RAM
# ==========================================
def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype

        if col_type != object and not isinstance(col_type, pd.CategoricalDtype):
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            elif str(col_type)[:5] == 'float':
                if c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        else:
            if col.endswith('_id') or col == 'id' or col == 'transaction_id' or 'date' in col:
                df[col] = df[col].astype(str)
            else:
                num_unique = len(df[col].unique())
                if num_unique / len(df) < 0.5:
                    df[col] = df[col].astype('category')

    end_mem = df.memory_usage().sum() / 1024**2
    print(f" -> Dung lượng RAM giảm từ {start_mem:.2f} MB xuống còn {end_mem:.2f} MB (Tiết kiệm {(start_mem - end_mem)/start_mem*100:.1f}%)")
    return df

# ==========================================
# HÀM ĐỌC VÀ LỌC CỘT (COLUMN PRUNING) CẢI TIẾN
# ==========================================
def load_and_prune_data():
    print("\n[Bước 1.1] Đang đọc và tối ưu hóa bộ nhớ từng tệp dữ liệu thô...")

    cols_users = ['id', 'current_age', 'yearly_income', 'total_debt', 'credit_score', 'latitude', 'longitude', 'lat', 'long']
    df_users = pd.read_csv(paths['users'], usecols=lambda x: x in cols_users)

    for col in ['yearly_income', 'total_debt']:
        if col in df_users.columns:
            df_users[col] = df_users[col].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

    df_users.rename(columns={'latitude': 'user_lat', 'longitude': 'user_lon', 'lat': 'user_lat', 'long': 'user_lon'}, inplace=True)
    df_users = reduce_mem_usage(df_users)

    cols_cards = ['id', 'client_id', 'card_type', 'credit_limit', 'card_on_dark_web', 'has_chip', 'acct_open_date', 'year_pin_last_changed']
    df_cards = pd.read_csv(paths['cards'], usecols=lambda x: x in cols_cards)

    if 'credit_limit' in df_cards.columns:
        df_cards['credit_limit'] = df_cards['credit_limit'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)
    df_cards = reduce_mem_usage(df_cards)

    cols_trans = ['id', 'date', 'client_id', 'card_id', 'amount', 'use_chip', 'mcc', 'errors', 'merchant_state', 'merchant_id', 'latitude', 'longitude', 'merch_lat', 'merch_long']
    df_transactions = pd.read_csv(paths['transactions'], usecols=lambda x: x in cols_trans)

    if df_transactions['amount'].dtype == 'object':
        df_transactions['amount'] = df_transactions['amount'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

    df_transactions.rename(columns={'latitude': 'merchant_lat', 'longitude': 'merchant_lon', 'merch_lat': 'merchant_lat', 'merch_long': 'merchant_lon'}, inplace=True)
    df_transactions = reduce_mem_usage(df_transactions)

    with open(paths['labels'], 'r') as f:
        labels_dict = json.load(f)
    if 'target' in labels_dict:
        labels_dict = labels_dict['target']

    df_labels = pd.DataFrame(list(labels_dict.items()), columns=['transaction_id', 'is_fraud'])

    label_mapping = {
        'yes': 1, 'no': 0, 'true': 1, 'false': 0, '1': 1, '0': 0, '1.0': 1, '0.0': 0
    }
    df_labels['is_fraud'] = df_labels['is_fraud'].astype(str).str.lower().str.strip().map(label_mapping).fillna(0).astype(np.int8)
    df_labels = reduce_mem_usage(df_labels)

    return df_users, df_cards, df_transactions, df_labels

# ==========================================
# LỚP FEATURE ENGINEERING NÂNG CAO (PRO-LEVEL)
# ==========================================
class RealDataFeatureEngineer:
    def __init__(self, df_users, df_cards, df_transactions, df_labels):
        self.df_users = df_users
        self.df_cards = df_cards
        self.df_transactions = df_transactions
        self.df_labels = df_labels

    def _join_data(self):
        card_pk = 'id' if 'id' in self.df_cards.columns else 'user_id'
        df = self.df_transactions.merge(self.df_cards, left_on='card_id', right_on=card_pk, how='left', suffixes=('', '_card'))
        del self.df_transactions
        gc.collect()

        user_pk = 'id' if 'id' in self.df_users.columns else 'card_id'
        df = df.merge(self.df_users, left_on='client_id', right_on=user_pk, how='left', suffixes=('', '_user'))
        del self.df_users
        gc.collect()

        df['id'] = df['id'].astype(str)
        self.df_labels['transaction_id'] = self.df_labels['transaction_id'].astype(str)
        df = df.merge(self.df_labels, left_on='id', right_on='transaction_id', how='left')
        df['is_fraud'] = df['is_fraud'].fillna(0).astype(np.int8)

        drop_cols = [c for c in ['id_card', 'id_user', 'transaction_id'] if c in df.columns]
        df.drop(columns=drop_cols, inplace=True, errors='ignore')

        return df

    def _haversine_distance(self, lat1, lon1, lat2, lon2):
        R = 6371
        lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
        dlat = lat2 - lat1
        dlon = lon2 - lon1
        a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
        c = 2 * np.arcsin(np.sqrt(a))
        return (R * c).astype(np.float32)

    def build_features(self):
        print("\n[Bước 2.1] Đang khớp nối dữ liệu thô (Merge & Join)...")
        df = self._join_data()

        print("[Bước 2.2] Đang chuyển đổi định dạng ngày tháng...")
        df['date'] = pd.to_datetime(df['date'], format='mixed', errors='coerce')

        # Tạo cột lỗi trước để sử dụng cho Merchant Risk
        df['is_error_txn'] = df['errors'].notnull().astype(np.int8)

        # --- GIAI ĐOẠN 1: TÍNH TOÁN THEO THẺ (CARD-BASED) ---
        print("[Bước 2.3] Sắp xếp theo Thẻ & Thời gian để tính Card Features...")
        df = df.sort_values(['card_id', 'date']).reset_index(drop=True)
        gc.collect()

        print(" -> Tính toán Amount Features...")
        df['flag_is_refund'] = (df['amount'] < 0).astype(np.int8)
        df['abs_amount'] = df['amount'].abs().astype(np.float32)
        df['net_amount'] = df['amount'].astype(np.float32)
        df['log_abs_amount'] = np.log1p(df['abs_amount']).astype(np.float32)

        print(" -> Tính toán Temporal & Velocity (Card)...")
        df['time_since_last_txn'] = df.groupby('card_id')['date'].diff().dt.total_seconds().fillna(999999).astype(np.float32)
        df['flag_rapid_sequence'] = (df['time_since_last_txn'] < 60).astype(np.int8)

        # Tạo index thời gian tạm thời để tính rolling cho Card
        df = df.set_index('date')
        df['txn_count_1h'] = df.groupby('card_id')['id'].rolling('1h', closed='left').count().reset_index(level=0, drop=True).values
        df['txn_count_1h'] = df['txn_count_1h'].fillna(0).astype(np.int16)
        df['flag_burst_1h'] = (df['txn_count_1h'] > 5).astype(np.int8)

        print(" -> Tính toán Amount Stats (Rolling 7d)...")
        rolling_7d = df.groupby('card_id')['abs_amount'].rolling('7d', closed='left')
        df['amount_mean_7d'] = rolling_7d.mean().reset_index(level=0, drop=True).values.astype(np.float32)
        df['amount_std_7d'] = rolling_7d.std().reset_index(level=0, drop=True).values.astype(np.float32)

        # Reset index để quay lại dạng bảng thông thường trước khi chuyển giai đoạn
        df = df.reset_index()

        # --- GIAI ĐOẠN 2: TÍNH TOÁN THEO ĐIỂM BÁN (MERCHANT-BASED) ---
        print("[Bước 2.4] Sắp xếp lại theo Điểm bán để tính Merchant Risk...")
        # Sắp xếp theo merchant_id và date để đảm bảo tính "monotonic" cho từng nhóm merchant
        df = df.sort_values(['merchant_id', 'date']).reset_index(drop=True)
        df = df.set_index('date') # Set lại index date để dùng rolling thời gian

        df['merchant_error_count_24h'] = df.groupby('merchant_id')['is_error_txn'].rolling('24h', closed='left').sum().reset_index(level=0, drop=True).values
        df['merchant_error_count_24h'] = df['merchant_error_count_24h'].fillna(0).astype(np.int16)
        df['flag_hot_merchant'] = (df['merchant_error_count_24h'] > 5).astype(np.int8)

        # Xóa cột trung gian
        df = df.reset_index()
        df.drop(columns=['is_error_txn'], inplace=True)

        # --- GIAI ĐOẠN 3: CÁC LOGIC CÒN LẠI ---
        print("[Bước 2.5] Hoàn thiện các đặc trưng bảo mật và địa lý...")

        # Cờ thời gian
        df['flag_night_txn'] = df['date'].dt.hour.between(0, 5).astype(np.int8)
        df['is_weekend'] = df['date'].dt.dayofweek.isin([5, 6]).astype(np.int8)

        # Z-score
        df['amount_zscore'] = np.where(df['amount_std_7d'] > 0, (df['abs_amount'] - df['amount_mean_7d']) / df['amount_std_7d'], 0).astype(np.float32)
        df['flag_amount_spike'] = (df['amount_zscore'] > 3.0).astype(np.int8)

        if 'credit_limit' in df.columns:
            df['flag_near_credit_limit'] = (df['abs_amount'] > (0.85 * df['credit_limit'])).astype(np.int8)
        else:
            df['flag_near_credit_limit'] = np.int8(0)

        df['flag_round_amount'] = (df['abs_amount'].isin([1, 5, 10, 20, 50, 100]) & (df['flag_is_refund'] == 0)).astype(np.int8)
        df['flag_high_mcc_risk'] = df['mcc'].isin([4829, 6011, 7995, 5912, 6051]).astype(np.int8)

        df['flag_dark_web_card'] = (df['card_on_dark_web'] == 'Yes').astype(np.int8)
        df['flag_chip_bypass'] = ((df['has_chip'] == 'YES') & (df['use_chip'] != 'Chip Transaction')).astype(np.int8)

        if 'acct_open_date' in df.columns:
            df['acct_open_date'] = pd.to_datetime(df['acct_open_date'], format='mixed', errors='coerce')
            df['days_since_open'] = (df['date'] - df['acct_open_date']).dt.days.fillna(9999).astype(np.int32)
            df['flag_new_card'] = (df['days_since_open'] < 90).astype(np.int8)

        current_year = datetime.now().year
        if 'year_pin_last_changed' in df.columns:
            df['flag_pin_stale'] = ((current_year - df['year_pin_last_changed']) > 5).astype(np.int8)

        df['prev_merchant_state'] = df.groupby('card_id')['merchant_state'].shift(1)
        df['flag_state_hop'] = ((df['merchant_state'] != df['prev_merchant_state']) & (df['prev_merchant_state'].notnull()) & (df['time_since_last_txn'] < 6 * 3600)).astype(np.int8)

        df['flag_international'] = (df['merchant_state'].isnull() | (df['merchant_state'] == 'FOREIGN')).astype(np.int8)
        df['flag_night_txn'] = ((df['flag_night_txn'] == 1) & (df['flag_international'] == 0)).astype(np.int8)

        if 'user_lat' in df.columns and 'merchant_lat' in df.columns:
            df['distance_km'] = self._haversine_distance(df['user_lat'], df['user_lon'], df['merchant_lat'], df['merchant_lon'])
            df['distance_km'] = df['distance_km'].fillna(-1).astype(np.float32)
            df['flag_far_from_home'] = (df['distance_km'] > 200).astype(np.int8)

            # Vận tốc vật lý (Impossible Travel)
            # Sắp xếp lại theo card_id để diff thời gian chính xác (nếu cần)
            # Nhưng ở đây ta đã có time_since_last_txn tính từ đầu rồi
            time_hours = (df['time_since_last_txn'] / 3600.0) + 0.0001
            df['travel_speed_kmh'] = (df['distance_km'] / time_hours).astype(np.float32)
            df['flag_impossible_travel'] = ((df['travel_speed_kmh'] > 1000) & (df['distance_km'] > 100)).astype(np.int8)

        df['is_new_merchant'] = (~df.duplicated(subset=['card_id', 'merchant_id'])).astype(np.int8)

        if {'total_debt', 'yearly_income', 'credit_score'}.issubset(df.columns):
            df['flag_debt_pressure'] = ((df['total_debt'] / df['yearly_income'] > 0.8) & (df['credit_score'] < 620)).astype(np.int8)
        else:
            df['flag_debt_pressure'] = np.int8(0)

        gc.collect()
        print("=> Hoàn tất tính toán các đặc trưng (Features)!")
        return df

# ==========================================
# CHƯƠNG TRÌNH CHÍNH (MAIN EXECUTION)
# ==========================================
if __name__ == "__main__":
    print("=== PIPELINE TẠO ĐẶC TRƯNG CHUYÊN SÂU ===")

    # 1. Đọc và tối ưu hóa bộ nhớ
    df_users, df_cards, df_transactions, df_labels = load_and_prune_data()

    # 2. Chạy Pipeline tạo Features
    pipeline = RealDataFeatureEngineer(df_users, df_cards, df_transactions, df_labels)
    final_df = pipeline.build_features()

    # Giải phóng RAM các biến tạm thô
    del df_users, df_cards, df_labels
    gc.collect()

    # 3. KIỂM TRA CHẤT LƯỢNG DỮ LIỆU CUỐI CÙNG
    print("\n" + "="*50)
    print("=== BÁO CÁO CHẤT LƯỢNG DỮ LIỆU CUỐI CÙNG (DATA QUALITY REPORT) ===")
    print("="*50)

    print(f"Tổng số bản ghi sau khi Join và Tạo Features: {final_df.shape[0]} dòng, {final_df.shape[1]} cột")

    # Kiểm tra phân phối nhãn
    fraud_distribution = final_df['is_fraud'].value_counts()
    fraud_ratio = final_df['is_fraud'].value_counts(normalize=True) * 100
    print("\n* Phân phối nhãn mục tiêu (is_fraud):")
    for val, count in fraud_distribution.items():
        label_name = "Gian lận (1)" if val == 1 else "Hợp pháp (0)"
        print(f"  - {label_name}: {count} dòng ({fraud_ratio[val]:.3f}%)")

    # 4. Lưu kết quả
    output_path = os.path.join(DATA_DIR, "featured_transactions_data1.csv")
    print(f"\n[Bước 3] Đang lưu tệp dữ liệu hoàn chỉnh ra file mới...")
    final_df.to_csv(output_path, index=False)
    print(f" -> Đã lưu thành công tại: {output_path}")

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import precision_recall_curve, auc
import matplotlib.pyplot as plt
import gc

DATA_PATH = r"/content/DSS/content/DSS/archive/featured_transactions_data1.csv"
DEBUG_MODE = True  # Đổi thành False khi muốn chạy trên toàn bộ 13 triệu dòng

def step1_load_all_data_safely(file_path):
    print("--- BƯỚC 1: NẠP TOÀN BỘ DỮ LIỆU (CƠ CHẾ CHỐNG TRÀN RAM) ---")

    # 1. Quét schema trước để nén RAM (int8, float32)
    sample = pd.read_csv(file_path, nrows=1000)
    dtypes_dict = {}
    cols_to_drop = [c for c in sample.columns if c.endswith('_id') or c in ['id', 'transaction_id', 'date', 'acct_open_date']]

    # 2. Đọc theo phân mảnh (Chunking) để lọc dữ liệu
    chunks = []
    # Đọc mỗi lần 1 triệu dòng
    for chunk in pd.read_csv(file_path, chunksize=1000000, usecols=lambda x: x not in cols_to_drop):
        # Giữ lại 100% gian lận
        fraud = chunk[chunk['is_fraud'] == 1]
        # Chỉ lấy 10% giao dịch hợp pháp để giảm tải RAM (nhưng vẫn đủ lớn để học)
        legit = chunk[chunk['is_fraud'] == 0].sample(frac=0.1, random_state=42)

        combined_chunk = pd.concat([fraud, legit])
        chunks.append(combined_chunk)
        print(f" -> Đã xử lý xong 1 triệu dòng...")

    df = pd.concat(chunks, ignore_index=True)

    # Ép kiểu lại lần cuối để cực kỳ nhẹ RAM
    for col in df.select_dtypes(include=['float', 'int']).columns:
        df[col] = pd.to_numeric(df[col], downcast='float')

    print(f" -> TỔNG CỘNG nạp vào RAM: {len(df)} dòng (Đã giữ toàn bộ gian lận + 10% hợp pháp)")
    return df

# Chạy lại Bước 1 thực thụ
df_raw = step1_load_all_data_safely(DATA_PATH)

In [ ]:
def step2_prepare_data(df):
    print("\n--- BƯỚC 2: CHUẨN BỊ DỮ LIỆU ---")

    # 1. Mã hóa Label Encoding cho các cột dạng chữ
    cat_cols = df.select_dtypes(include=['object']).columns
    if len(cat_cols) > 0:
        print(f" -> Đang mã hóa {len(cat_cols)} cột văn bản: {list(cat_cols)}")
        for col in cat_cols:
            df[col] = df[col].astype('category').cat.codes

    # 2. Xử lý giá trị khuyết (NaN) - Mặc dù XGBoost tự xử lý được NaN, nhưng điền trước sẽ an toàn hơn
    # Điền NaN bằng -999 để XGBoost hiểu đây là một nhóm đặc biệt
    df = df.fillna(-999)

    # 3. Tách X, y
    X = df.drop(columns=['is_fraud'])
    y = df['is_fraud'].astype(np.int8)

    # 4. Chia Train/Test (80/20) - Mặc định dữ liệu đã xếp theo ngày tháng từ bước Feature Engineering
    split_idx = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    print(f" -> Chia dữ liệu thành công! Train: {len(X_train)} dòng | Test: {len(X_test)} dòng")

    # Dọn RAM
    del df
    gc.collect()

    return X_train, X_test, y_train, y_test

# Chạy thử Bước 2
X_train, X_test, y_train, y_test = step2_prepare_data(df_raw)

In [ ]:
def step3_train_xgboost(X_train, X_test, y_train, y_test):
    print("\n--- BƯỚC 3: HUẤN LUYỆN XGBOOST (TỐI ƯU CHO 1.3 TRIỆU DÒNG) ---")

    # Tính trọng số mất cân bằng lớp
    # Lúc này tỷ lệ đã giảm xuống khoảng 1:123 (do ta chỉ lấy 10% hợp pháp ở bước 1)
    spw = (len(y_train) - sum(y_train)) / (sum(y_train) + 1e-5)

    # Mẹo: Với dữ liệu lớn, spw tính toán đôi khi quá gắt làm tăng False Positives.
    # Tạm thời ta lấy spw gốc, nhưng nếu mô hình bắt nhầm nhiều, bạn có thể sửa thành: spw = spw / 2
    print(f" -> Lớp gian lận đã cân bằng hơn. Kích hoạt scale_pos_weight = {spw:.2f}")

    # Cấu hình XGBoost chống Overfitting và Tối ưu RAM
    xgb_params = {
        'n_estimators': 500,        # [SỬA] Tăng lên 500 cây (vì đã có Early stopping chặn lại)
        'learning_rate': 0.05,
        'max_depth': 4,             # [SỬA] Giảm từ 6 xuống 4 để chống học vẹt
        'min_child_weight': 20,     # [SỬA] Tăng số lượng mẫu tối thiểu ở lá để tránh nhiễu
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 5.0,           # [MỚI] Hình phạt L1: Tự động loại bỏ feature vô dụng
        'reg_lambda': 5.0,          # [MỚI] Hình phạt L2: Kìm hãm trọng số quá lớn
        'scale_pos_weight': spw,
        'tree_method': 'hist',
        'eval_metric': 'aucpr',     # [MỚI] Tối ưu hóa trực tiếp trên điểm PR-AUC
        'early_stopping_rounds': 50,# [MỚI] Phanh an toàn: Dừng nếu 50 vòng không tiến bộ
        'random_state': 42,
        'n_jobs': -1
    }

    # Ở các bản XGBoost mới, early_stopping_rounds được truyền vào XGBClassifier
    model = xgb.XGBClassifier(**xgb_params)

    print(" -> Bắt đầu huấn luyện với Early Stopping...")
    # Dùng eval_set để theo dõi tiến trình trên tập Test
    model.fit(
        X_train, y_train,
        eval_set=[(X_train, y_train), (X_test, y_test)],
        verbose=50 # Cứ 50 cây thì in kết quả ra màn hình 1 lần
    )

    print(f" -> Huấn luyện hoàn tất! Cây tốt nhất dừng ở vòng: {model.best_iteration}")
    return model

# Chạy thử Bước 3
model_xgb = step3_train_xgboost(X_train, X_test, y_train, y_test)

In [ ]:
import seaborn as sns

def step4_evaluate(model, X_train, X_test, y_train, y_test):
    print("\n--- BƯỚC 4: ĐÁNH GIÁ MÔ HÌNH ---")

    # 1. Dự đoán xác suất
    y_test_proba = model.predict_proba(X_test)[:, 1]
    y_train_proba = model.predict_proba(X_train)[:, 1]

    # 2. Tính PR-AUC
    p_test, r_test, thresholds = precision_recall_curve(y_test, y_test_proba)
    pr_auc_test = auc(r_test, p_test)

    p_train, r_train, _ = precision_recall_curve(y_train, y_train_proba)
    pr_auc_train = auc(r_train, p_train)

    print(f" -> [ĐIỂM SỐ] PR-AUC (Tập Train): {pr_auc_train:.4f}")
    print(f" -> [ĐIỂM SỐ] PR-AUC (Tập Test) : {pr_auc_test:.4f}")

    # 3. Tìm Ngưỡng (Threshold) tối ưu hóa F1-Score
    f1_scores = 2 * (p_test * r_test) / (p_test + r_test + 1e-10)
    optimal_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[optimal_idx] if optimal_idx < len(thresholds) else 0.5

    print(f"\n=> Ngưỡng cảnh báo tối ưu (Optimal Threshold): {optimal_threshold:.4f}")

    # 4. Báo cáo phân loại
    y_pred_optimal = (y_test_proba >= optimal_threshold).astype(int)
    from sklearn.metrics import classification_report, confusion_matrix
    print("\nBÁO CÁO PHÂN LOẠI:")
    print(classification_report(y_test, y_pred_optimal))

    # 5. Vẽ Confusion Matrix
    cm = confusion_matrix(y_test, y_pred_optimal)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.xlabel('Dự đoán (Predicted)')
    plt.ylabel('Thực tế (Actual)')
    plt.title(f'Confusion Matrix (Threshold = {optimal_threshold:.2f})')
    plt.show()

    # 6. Trực quan hóa Feature Importance
    importance = model.feature_importances_
    feat_imp = pd.DataFrame({'Feature': X_train.columns, 'Importance': importance})
    feat_imp = feat_imp.sort_values(by='Importance', ascending=False).head(15)

    plt.figure(figsize=(10, 6))
    plt.barh(feat_imp['Feature'][::-1], feat_imp['Importance'][::-1], color='teal')
    plt.title("Top 15 Đặc trưng quyết định Gian lận (XGBoost)")
    plt.xlabel("Trọng số quan trọng")
    plt.show()

# Chạy Bước 4
step4_evaluate(model_xgb, X_train, X_test, y_train, y_test)